# Week05 Part 2 — 공개 사전학습 모델을 이용한 전이학습

## 목표
- Teachable Machine 모델이 없어도 실습할 수 있다.
- ImageNet 사전학습 MobileNetV2를 특징 추출기로 사용한다.
- Feature Extraction과 Fine-tuning 결과를 비교한다.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.squeeze()
y_test = y_test.squeeze()
selected = [0, 1, 2]
class_names = ["airplane", "automobile", "bird"]
train_mask = np.isin(y_train, selected)
test_mask = np.isin(y_test, selected)
x_train = x_train[train_mask][:4500]
y_train = y_train[train_mask][:4500]
x_test = x_test[test_mask][:900]
y_test = y_test[test_mask][:900]


In [ ]:
IMG_SIZE = 96
BATCH_SIZE = 32
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(2000).map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(3, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_ds, validation_data=test_ds, epochs=3)


In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 25
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
fine_history = model.fit(train_ds, validation_data=test_ds, epochs=2)


In [ ]:
loss, acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {acc:.4f}")
model.save("week05_part2_mobilenetv2.keras")


## 체크리스트
- [ ] 공개 사전학습 모델 불러오기
- [ ] base model 동결
- [ ] 새 분류기 학습
- [ ] Fine-tuning
- [ ] 전/후 결과 비교
- [ ] 오분류 사례 분석
